# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² tabular dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL, which describes the records and structure.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\n\nDescription: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).
The `@id` uniquely identifies each dataset entity. We'll enumerate all record sets and their fields using the dataset's Croissant metadata.


In [ ]:
# List all record sets with their @id and fields
print("Available Record Sets and Fields (all IDs are @id):\n")

record_sets_info = dataset.metadata.record_sets
if not record_sets_info:
    print("No record sets found in metadata. Attempting to auto-discover via dataset.records().")
    # Some Croissant schemas do not list record sets directly in JSON-LD.
    # Instead, try to enumerate via internal Dataset config
    discovered_record_sets = dataset._config.get('record_sets', {}).keys()
    for rs_id in discovered_record_sets:
        print(f"- Record Set @id: {rs_id}")
        # Try to fetch its fields
        rs_obj = dataset._config['record_sets'][rs_id]
        fields = rs_obj.get('fields', {})
        if isinstance(fields, dict):
            for field_id, field_info in fields.items():
                print(f"    - Field @id: {field_id}")
        elif isinstance(fields, list):
            for field in fields:
                print(f"    - Field @id: {field['@id']}")
else:
    # Standard Croissant: use metadata methods
    for rs in record_sets_info:
        print(f"- Record Set @id: {rs.id}, name: {getattr(rs, 'name', '')}")
        for field in rs.fields:
            print(f"    - Field @id: {field.id}, name: {getattr(field, 'name', '')}")

### Sample record preview
Let's preview the first record from each record set. 

In [ ]:
# List detected record set @ids
record_set_ids = list(dataset._config.get('record_sets', {}).keys())
if record_set_ids:
    for rs_id in record_set_ids:
        print(f"\nFirst record from record set @id: {rs_id}")
        records_iter = dataset.records(record_set=rs_id)
        try:
            record = next(records_iter)
            pprint.pprint(record)
        except StopIteration:
            print("  (No records)")
else:
    print("No record sets found.")

## 3. Data Extraction
Load the data from record sets into pandas DataFrames.

**Note:** All references to record sets, fields, and columns use their full Croissant `@id`s.

In [ ]:
# Extract data from all record sets by @id
record_set_ids = list(dataset._config.get('record_sets', {}).keys())
dataframes = {}

for rs_id in record_set_ids:
    try:
        df = pd.DataFrame(list(dataset.records(record_set=rs_id)))
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records from record set @id: {rs_id}")
    except Exception as e:
        print(f"Failed to load records from {rs_id}: {e}")

# For demonstration, pick the first record set for analysis:
if record_set_ids:
    primary_record_set_id = record_set_ids[0]
    print(f"\nFields (columns) in record set {primary_record_set_id}:")
    print(dataframes[primary_record_set_id].columns.tolist())
    display(dataframes[primary_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
We'll demonstrate numeric filtering, normalization, and grouping operations on a selected numeric field referenced by its `@id`.

### Steps:
- Choose a numeric field by inspecting displayed columns (look for any columns suggesting numeric values, e.g., 'age', 'interval', 'metastasis', etc.).
- Filter for values above a threshold
- Normalize
- Group by a categorical field (e.g., 'Sex' or 'MSI status')

*Adjust these column `@id`s as appropriate for the dataset structure.*

In [ ]:
# List columns from the main record set
df = dataframes[primary_record_set_id]
print("Available columns (all are field @ids):")
print(list(df.columns))

# Choose column @ids based on data dictionary or inspection
# For demonstration, let's hypothetically choose the following (replace as appropriate):
# Assume the following @ids or column names exist—adjust as needed by inspecting df.columns
numeric_field_id = None
for col in df.columns:
    if 'age' in col.lower():
        numeric_field_id = col
        break
if not numeric_field_id:
    # Fallback: pick the first numeric column found
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

if numeric_field_id:
    print(f"\nUsing numeric field @id: {numeric_field_id}")

    # Set an example threshold
    threshold = 50
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[numeric_field_id + '_normalized'] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())
        / filtered_df[numeric_field_id].std())
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    # Pick a grouping field (categorical, e.g., sex/male/female or anatomical site)
    group_field = None
    for col in df.columns:
        if any(word in col.lower() for word in ['sex', 'site', 'location', 'msi']):
            group_field = col
            break

    if group_field:
        # Only group numeric columns (groupby + mean())
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped by field @id: {group_field} (mean values):")
        display(grouped_df.head())
    else:
        print("No suitable grouping field found in columns.")
else:
    print("No numeric field found for analysis. Please inspect columns and update `numeric_field_id`.")

## 5. Visualization
We'll visualize the distribution of the selected numeric field and, if available, its relationship with the chosen group field. All axes and legends refer to column `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of numeric field (@id): {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    if group_field:
        plt.figure(figsize=(7,5))
        sns.boxplot(x=df[group_field], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} grouped by {group_field} (@ids)")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric field available to plot.")

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR² dataset using the `mlcroissant` library. By referencing all dataset entities (record sets, fields, columns) by their Croissant `@id`, we ensured consistent and schema-aligned access to the data. Key steps included loading and inspecting the metadata, extracting records, basic exploratory data analysis (filtering, normalization, grouping), and visualizing field distributions. 

For further analysis, adjust the field or grouping `@id`s to target the most relevant dataset attributes for your use case.